# ЛР2. Обучение GPT-like модели
GPT-декодер с нуля (PyTorch+Lightning), данные/токенизатор из ЛР1.

In [ ]:
import os, sys
# ROOT задаётся драйвером на Kaggle (/kaggle/working); локально — текущая папка проекта
ROOT = os.environ.get('LAB2_ROOT', os.path.abspath('..'))
sys.path.insert(0, ROOT); os.chdir(ROOT)
import torch, lightning as L
from omegaconf import OmegaConf
from src.utils import load_config, env
print('device cuda:', torch.cuda.is_available())

## Конфиг (всё из YAML; SMOKE — быстрый прогон для проверки пайплайна)

In [ ]:
cfg = load_config(os.environ.get('LAB2_CONFIG','configs/model.yaml'))
SMOKE = os.environ.get('LAB2_SMOKE','0') == '1'
if SMOKE:
    cfg.data.max_documents = 3000
    cfg.model.n_layers, cfg.model.d_model, cfg.model.d_ff = 4, 256, 1024
    cfg.optim.max_steps, cfg.optim.warmup_steps = 150, 30
    cfg.trainer.val_check_interval = 75
print(OmegaConf.to_yaml(cfg))
if os.environ.get('LAB2_QUICK','0')=='1':
    cfg.optim.max_steps=2500; cfg.optim.warmup_steps=200
    cfg.trainer.val_check_interval=250; cfg.trainer.track_layer_grad_norms=True
    print('QUICK ClearML run: полная модель, 2500 шагов')

## Данные: wikitext + BPE из ЛР1 -> packed-батчи

In [ ]:
from src.data.datamodule import LMDataModule
dm = LMDataModule(cfg); dm.setup()
cfg.model.vocab_size = dm.vocab_size
print('vocab:', dm.vocab_size, '| train batches:', len(dm.train_ds), '| val batches:', len(dm.val_ds))
b = dm.train_ds[0]; print('input_ids', b['input_ids'].shape, '| segment_ids уникальные:', b['segment_ids'].unique().tolist()[:6])

## Модель

In [ ]:
from src.training.lightning_module import GPTLitModule
model = GPTLitModule(cfg, dm.vocab_size)
print('параметров:', sum(p.numel() for p in model.parameters())/1e6, 'M')

## Обучение (ClearML/TensorBoard, warmup+cosine, grad-clip, чекпоинты)

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from src.training.callbacks import GradNormCallback
try:
    from clearml import Task; Task.init(project_name=cfg.logging.clearml_project, task_name=cfg.logging.clearml_task)
except Exception as e:
    print('ClearML off:', e)
ckpt_dir = env('CHECKPOINT_DIR','./checkpoints')
ckpt = ModelCheckpoint(dirpath=ckpt_dir, monitor='val_perplexity', mode='min', save_top_k=1, save_last=True, filename='gpt-{step}-{val_perplexity:.2f}')
trainer = L.Trainer(max_steps=cfg.optim.max_steps, precision=cfg.trainer.precision,
    accumulate_grad_batches=cfg.trainer.accumulate_grad_batches, val_check_interval=cfg.trainer.val_check_interval,
    log_every_n_steps=cfg.trainer.log_every_n_steps, max_time=cfg.trainer.get('max_time'), gradient_clip_val=cfg.optim.grad_clip, gradient_clip_algorithm='norm',
    logger=TensorBoardLogger(ckpt_dir, name='tb'), callbacks=[ckpt, LearningRateMonitor('step'), GradNormCallback(cfg.trainer.track_layer_grad_norms)])
trainer.fit(model, dm, ckpt_path=os.environ.get('RESUME_CKPT') or None)

## Перплексия на валидации

In [ ]:
trainer.validate(model, dm)
print('val_perplexity =', float(trainer.callback_metrics.get('val_perplexity', float('nan'))))

## Генерация в режиме инференса

In [ ]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file(cfg.data.bpe_tokenizer_path)
ids = torch.tensor([tok.encode(cfg.generate.prompt).ids], device=model.device)
out = model.generate(ids, max_new_tokens=cfg.generate.max_new_tokens, temperature=cfg.generate.temperature, top_k=cfg.generate.top_k)
print('PROMPT:', cfg.generate.prompt)
print('GEN   :', tok.decode(out[0].tolist()))